# Exploratory Data Analysis — Demand Patterns

**Objective:** Understand demand patterns in the Urban Black ride-sharing service

This notebook covers:
1. Data loading and basic statistics
2. Temporal patterns (hourly, daily, weekly)
3. Geographic distribution (zones)
4. Ride characteristics
5. Data quality assessment

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ Libraries loaded successfully')

## 1. Load Data

In [ ]:
# Load raw data
print('Loading data...')
rides = pd.read_parquet('data/raw/rides_6months.parquet')

print(f'✅ Loaded {len(rides):,} rides')
print(f'\nColumns: {rides.columns.tolist()}')
print(f'\nData types:\n{rides.dtypes}')

## 2. Basic Statistics

In [ ]:
# Convert to datetime
rides['requestedAt'] = pd.to_datetime(rides['requestedAt'])

# Basic stats
print(f'Time range: {rides["requestedAt"].min()} to {rides["requestedAt"].max()}')
print(f'Total rides: {len(rides):,}')
print(f'Unique dates: {rides["requestedAt"].dt.date.nunique()}')
print(f'\nRide status distribution:')
print(rides['status'].value_counts())
print(f'\nCompletion rate: {(rides["status"] == "COMPLETED").mean():.1%}')

## 3. Temporal Analysis

In [ ]:
# Extract temporal features
rides['hour'] = rides['requestedAt'].dt.hour
rides['day_name'] = rides['requestedAt'].dt.day_name()
rides['day_of_week'] = rides['requestedAt'].dt.dayofweek
rides['date'] = rides['requestedAt'].dt.date

# Hourly demand
hourly_demand = rides.groupby('hour').size()

print('Rides by hour:')
print(hourly_demand)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Hourly pattern
hourly_demand.plot(ax=axes[0,0], title='Demand by Hour of Day', marker='o')
axes[0,0].set_ylabel('Ride Count')
axes[0,0].set_xlabel('Hour of Day')

# Daily pattern
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_demand = rides.groupby('day_name').size().reindex(day_order)
daily_demand.plot(kind='bar', ax=axes[0,1], title='Demand by Day of Week', color='steelblue')
axes[0,1].set_ylabel('Ride Count')
axes[0,1].tick_params(axis='x', rotation=45)

# Weekday vs Weekend
rides['is_weekend'] = rides['day_of_week'].isin([5, 6])
weekend_demand = rides.groupby('is_weekend').size()
weekend_demand.index = ['Weekday', 'Weekend']
weekend_demand.plot(kind='bar', ax=axes[1,0], title='Weekday vs Weekend Demand', color=['green', 'red'])
axes[1,0].set_ylabel('Ride Count')
axes[1,0].tick_params(axis='x', rotation=0)

# Time series
daily_ts = rides.groupby('date').size()
daily_ts.plot(ax=axes[1,1], title='Daily Demand Over Time', color='purple', alpha=0.7)
axes[1,1].set_ylabel('Rides per Day')
axes[1,1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('plots/01_temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Temporal analysis complete')

## 4. Geographic Analysis

In [ ]:
# Map to zones using geohash
import geohash2

def get_zone(lat, lng, precision=5):
    try:
        return geohash2.encode(lat, lng, precision=precision)
    except:
        return None

rides['zone_id'] = rides.apply(
    lambda row: get_zone(row['pickupLat'], row['pickupLng']),
    axis=1
)

valid_zones = rides[rides['zone_id'].notna()]
print(f'Rides with valid zones: {len(valid_zones):,} / {len(rides):,}')
print(f'Unique zones: {valid_zones["zone_id"].nunique()}')

# Top zones
zone_counts = valid_zones['zone_id'].value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top zones
zone_counts.plot(kind='barh', ax=axes[0], title='Top 20 Zones by Ride Count')
axes[0].set_xlabel('Number of Rides')

# Zone distribution
zone_counts_all = valid_zones['zone_id'].value_counts()
axes[1].hist(zone_counts_all.values, bins=50, title='Distribution of Rides per Zone', color='steelblue', edgecolor='black')
axes[1].set_xlabel('Rides per Zone')
axes[1].set_ylabel('Number of Zones')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('plots/02_geographic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Geographic analysis complete')

## 5. Ride Characteristics

In [ ]:
# Analyze ride characteristics
rides_clean = rides[rides['durationMin'].notna() & (rides['durationMin'] > 0) & (rides['durationMin'] < 200)]

print('Ride duration statistics:')
print(rides_clean['durationMin'].describe())

print('\nFare statistics:')
print(rides_clean['fare'].describe())

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Duration distribution
rides_clean['durationMin'].hist(bins=50, ax=axes[0,0], title='Ride Duration Distribution', edgecolor='black')
axes[0,0].set_xlabel('Duration (minutes)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(rides_clean['durationMin'].median(), color='red', linestyle='--', label='Median')
axes[0,0].legend()

# Fare distribution
rides_clean[rides_clean['fare'] > 0]['fare'].hist(bins=50, ax=axes[0,1], title='Fare Distribution', edgecolor='black')
axes[0,1].set_xlabel('Fare ($)')
axes[0,1].set_ylabel('Frequency')

# Duration by hour
rides_clean.groupby('hour')['durationMin'].mean().plot(ax=axes[1,0], marker='o', title='Average Duration by Hour')
axes[1,0].set_ylabel('Duration (minutes)')
axes[1,0].set_xlabel('Hour')

# Fare by hour
rides_clean.groupby('hour')['fare'].mean().plot(ax=axes[1,1], marker='o', color='orange', title='Average Fare by Hour')
axes[1,1].set_ylabel('Fare ($)')
axes[1,1].set_xlabel('Hour')

plt.tight_layout()
plt.savefig('plots/03_ride_characteristics.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Ride characteristics analysis complete')

## 6. Demand Seasonality

In [ ]:
# Hour-of-day seasonality
hourly_pattern = rides.groupby('hour').size() / rides.groupby('hour').size().sum()
hourly_seasonal_index = hourly_pattern / hourly_pattern.mean()

print('Hourly seasonal indices (normalized hourly demand):')
for hour, idx in hourly_seasonal_index.items():
    print(f'  Hour {hour:2d}: {idx:.2f}')

# Day-of-week seasonality
daily_pattern = rides.groupby('day_name').size().reindex(day_order) / rides.groupby('day_name').size().sum()
daily_seasonal_index = daily_pattern / daily_pattern.mean()

print('\nDay-of-week seasonal indices:')
for day, idx in daily_seasonal_index.items():
    print(f'  {day:9s}: {idx:.2f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly_seasonal_index.plot(ax=axes[0], marker='o', linewidth=2, markersize=8)
axes[0].axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Hourly Seasonal Index')
axes[0].set_ylabel('Index (1.0 = average)')
axes[0].set_xlabel('Hour')
axes[0].grid(True, alpha=0.3)

daily_seasonal_index.reindex(day_order).plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Daily Seasonal Index')
axes[1].set_ylabel('Index (1.0 = average)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('plots/04_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Seasonality analysis complete')

## 7. Data Quality Assessment

In [ ]:
# Missing values
print('Missing values:')
missing = rides.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('  No missing values')
else:
    print(missing)

# Data quality metrics
print('\nData quality metrics:')
print(f'  Rides with valid lat/lng: {(rides["pickupLat"].notna() & rides["pickupLng"].notna()).sum():,} / {len(rides):,}')
print(f'  Completed rides: {(rides["status"] == "COMPLETED").sum():,} / {len(rides):,}')
print(f'  Rides with duration: {rides["durationMin"].notna().sum():,} / {len(rides):,}')
print(f'  Rides with fare: {rides["fare"].notna().sum():,} / {len(rides):,}')

# Outliers
print('\nOutlier analysis:')
duration_q1, duration_q3 = rides['durationMin'].quantile([0.25, 0.75])
duration_iqr = duration_q3 - duration_q1
duration_outliers = ((rides['durationMin'] < duration_q1 - 1.5*duration_iqr) | 
                     (rides['durationMin'] > duration_q3 + 1.5*duration_iqr)).sum()
print(f'  Duration outliers: {duration_outliers:,} ({duration_outliers/len(rides)*100:.1f}%)')

fare_q1, fare_q3 = rides[rides['fare'] > 0]['fare'].quantile([0.25, 0.75])
fare_iqr = fare_q3 - fare_q1
fare_outliers = ((rides['fare'] < fare_q1 - 1.5*fare_iqr) | 
                 (rides['fare'] > fare_q3 + 1.5*fare_iqr)).sum()
print(f'  Fare outliers: {fare_outliers:,} ({fare_outliers/len(rides)*100:.1f}%)')

# Summary
print('\n✅ Data quality assessment complete')

## 8. Summary & Key Findings

In [ ]:
print('\n' + '='*70)
print('KEY FINDINGS FROM EXPLORATORY DATA ANALYSIS')
print('='*70)

print(f'\n📊 DATA VOLUME')
print(f'  • Total rides: {len(rides):,}')
print(f'  • Time period: {rides["requestedAt"].min().date()} to {rides["requestedAt"].max().date()}')
print(f'  • Unique zones: {valid_zones["zone_id"].nunique()}')

print(f'\n⏰ TEMPORAL PATTERNS')
peak_hour = hourly_demand.idxmax()
print(f'  • Peak demand hour: {peak_hour}:00 ({hourly_demand[peak_hour]:,} rides)')
print(f'  • Low demand hour: {hourly_demand.idxmin()}:00 ({hourly_demand.min():,} rides)')
peak_day = daily_demand.idxmax()
print(f'  • Peak demand day: {peak_day} ({daily_demand[peak_day]:,} rides)')

print(f'\n📍 GEOGRAPHIC DISTRIBUTION')
print(f'  • Top zone: {zone_counts.index[0]} ({zone_counts.iloc[0]:,} rides)')
print(f'  • Concentration: Top 10 zones = {zone_counts.head(10).sum() / len(valid_zones) * 100:.1f}% of demand')

print(f'\n🚗 RIDE CHARACTERISTICS')
print(f'  • Avg duration: {rides_clean["durationMin"].mean():.1f} minutes')
print(f'  • Avg fare: ${rides_clean[rides_clean["fare"] > 0]["fare"].mean():.2f}')
print(f'  • Completion rate: {(rides["status"] == "COMPLETED").mean():.1%}')

print(f'\n✅ RECOMMENDATIONS')
print(f'  • Focus model on peak hours (8-10, 17-20)')
print(f'  • Handle day-of-week seasonality in features')
print(f'  • Consider zone-specific models for high-demand zones')
print(f'  • Account for completion rate variations')

print('\n' + '='*70)